# DNA Browser Test with new_jh_weights

This notebook demonstrates the enhanced DNA browser functionality using the `new_jh_weights` DNA vector from constants.py. It showcases the new missed scoring visualization features.

In [ ]:
# Import required libraries
import numpy as np
import sys
import os
from pathlib import Path

# Import project modules
from src.constants import (
    new_jh_weights, ACTIVE_SYNAPSES, NEURON_NAMES, TMAX, BIN_SIZE,
    CRITERIA_NAMES, INHIBITORY_NEURONS
)
from src.validation import diagnose_conditions
from dna_browser import create_dual_dna_browser
from dna_visualization import create_voltage_plot

print("✅ All imports successful")
print(f"📊 new_jh_weights contains {len(new_jh_weights)} connections")
print(f"🧬 ACTIVE_SYNAPSES contains {len(ACTIVE_SYNAPSES)} possible connections")

In [ ]:
def convert_weights_to_dna_vector(weight_connections, active_synapses):
    """Convert connection weights list to DNA vector format."""
    # Create a dictionary for fast lookup
    weight_dict = {(pre, post): weight for pre, post, weight in weight_connections}
    
    # Create DNA vector matching ACTIVE_SYNAPSES order
    dna_vector = []
    for pre, post in active_synapses:
        weight = weight_dict.get((pre, post), 0)
        # Ensure positive values for DNA encoding (negative weights handled by neuron types)
        dna_vector.append(abs(weight))
    
    return np.array(dna_vector, dtype=np.int32)

# Convert new_jh_weights to DNA vector format
dna_vector = convert_weights_to_dna_vector(new_jh_weights, ACTIVE_SYNAPSES)

print(f"🔬 DNA Vector Analysis:")
print(f"  Total connections: {len(dna_vector)}")
print(f"  Non-zero weights: {np.count_nonzero(dna_vector)}")
print(f"  Weight range: {dna_vector.min()} - {dna_vector.max()}")
print(f"  Average weight: {dna_vector[dna_vector > 0].mean():.1f}")

In [ ]:
def create_real_simulation_results(dna_vector):
    """Create REAL simulation results using the actual simulation engine."""
    from dna_simulation import run_dna_with_voltage_tracking
    
    print("🔬 Running REAL simulation with new_jh_weights...")
    try:
        results = run_dna_with_voltage_tracking(dna_vector)
        print("✅ Real simulation completed successfully")
        return results
    except Exception as e:
        print(f"❌ Real simulation failed: {e}")
        # Return empty structure if simulation fails
        voltages = {neuron_name: np.full(TMAX, -60.0, dtype=np.float32) for neuron_name in NEURON_NAMES}
        spike_raster = np.zeros((len(NEURON_NAMES), TMAX), dtype=np.uint8)
        
        return {
            'experimental': {
                'voltages': voltages,
                'missed_points': [],
                'spike_raster': spike_raster,
                'score': 0
            },
            'control': {
                'voltages': voltages,
                'missed_points': [],
                'spike_raster': spike_raster,
                'score': 0
            }
        }

# Generate REAL simulation data (no noise!)
print("🔬 Generating REAL simulation data...")
simulation_result = create_real_simulation_results(dna_vector)

exp_missed = len(simulation_result['experimental']['missed_points'])
cont_missed = len(simulation_result['control']['missed_points'])
total_score = simulation_result['experimental']['score'] + simulation_result['control']['score']

print(f"📊 REAL Simulation Results:")
print(f"  Experimental score: {simulation_result['experimental']['score']} (missed: {exp_missed})")
print(f"  Control score: {simulation_result['control']['score']} (missed: {cont_missed})")
print(f"  Total score: {total_score}")
print(f"  Data format: Matches analyze_multiple_ga_results.ipynb exactly")

In [ ]:
# Display the new_jh_weights connections
print("🔗 new_jh_weights Network Connections:")
print("=" * 50)

for i, (pre, post, weight) in enumerate(new_jh_weights):
    is_inhibitory = pre in INHIBITORY_NEURONS
    conn_type = "Inhibitory" if is_inhibitory else "Excitatory"
    print(f"{i+1:2d}. {pre:8s} → {post:8s} | {weight:6d} | {conn_type}")

print(f"\n📈 Network Properties:")
print(f"  Total connections: {len(new_jh_weights)}")
print(f"  Excitatory: {len([c for c in new_jh_weights if c[0] not in INHIBITORY_NEURONS])}")
print(f"  Inhibitory: {len([c for c in new_jh_weights if c[0] in INHIBITORY_NEURONS])}")
print(f"  Weight range: {min(w for _, _, w in new_jh_weights)} to {max(w for _, _, w in new_jh_weights)}")

In [ ]:
def create_real_dna_info(dna_id, dna_vector, simulation_results):
    """Create DNA info structure matching analyze_multiple_ga_results.ipynb format."""
    nonzero_weights = np.count_nonzero(dna_vector)
    
    exp_score = simulation_results['experimental']['score']
    cont_score = simulation_results['control']['score']
    total_score = exp_score + cont_score
    
    return {
        'id': dna_id,
        'original_dna': {
            'run_folder': 'test_new_jh_weights_real',
            'generation': 1,
            'process_id': 1,
            'original_dna_id': f'new_jh_weights_{dna_id}'
        },
        'pruned_dna': dna_vector,
        'original_score': total_score,
        'pruned_score': total_score,
        'original_nonzero': nonzero_weights,
        'pruned_nonzero': nonzero_weights,
        'weights_removed': 0,  # No pruning applied to new_jh_weights
        'final_exp_score': exp_score,
        'final_cont_score': cont_score,
    }

# Create REAL DNA info using actual simulation results
dna_info = create_real_dna_info(1, dna_vector, simulation_result)
target_dnas = [dna_info]
simulation_results = [simulation_result]

print("🧬 REAL DNA Info Created:")
print(f"  DNA ID: {dna_info['id']}")
print(f"  Scores: Original={dna_info['original_score']}, Pruned={dna_info['pruned_score']}")
print(f"  Weights: {dna_info['original_nonzero']} (no pruning applied)")
print(f"  Data format: Exactly matches analyze_multiple_ga_results.ipynb")

## Test Missed Scoring Visualization - REAL Data

The following cell will create a voltage plot using **REAL simulation data** (no artificial noise) with the new missed scoring features enabled. Look for:

- **🎯 Gold markers**: Criteria neurons are highlighted
- **🟠 Orange regions**: Time periods where scoring criteria were missed (from actual simulation)
- **📝 Annotations**: "Miss: W{wanted} G{got}" showing expected vs actual behavior
- **📊 Title**: Total missed points from real diagnostic analysis
- **⚡ Real voltage traces**: Authentic Izhikevich neuron dynamics

In [ ]:
# Test voltage plot with REAL missed scoring visualization
print("📈 Creating voltage plot with REAL missed scoring visualization...")

voltage_plot = create_voltage_plot(
    results=simulation_result,
    dna_info=dna_info,
    condition="experimental",
    show_missed_scoring=True
)

print("\n✅ REAL voltage plot created!")
print("Features from REAL simulation data:")
print("  🎯 Criteria neurons marked with gold symbols")
print("  🟠 Orange highlighting on ACTUAL missed scoring periods")
print("  📝 Real annotations showing expected vs actual from diagnose_conditions()")
print("  📊 Actual missed points count in the title")
print("  ⚡ Authentic Izhikevich neuron voltage dynamics")
print("  🔬 No artificial noise - pure simulation results")

## Interactive DNA Browser

The enhanced DNA browser includes:

### 🎛️ Controls
- **Sort dropdown**: Change ordering of DNAs
- **Show dropdown**: Switch between voltage traces, network graph, or both
- **Condition dropdown**: View experimental, control, or both conditions
- **DNA slider**: Browse through different DNA solutions

### 🔍 Missed Scoring Features
- **Automatic analysis**: Missed scoring is automatically enabled
- **Visual feedback**: Orange highlights show exactly where points were lost
- **Detailed diagnostics**: Annotations show expected vs actual neuron behavior

In [ ]:
# Create the enhanced DNA browser
print("🎛️ Creating DNA Browser with missed scoring visualization...")

browser = create_dual_dna_browser(
    target_dnas=target_dnas,
    pruned_results=[],  # Empty since we're using target_dnas
    simulation_results=simulation_results,
    pruning_threshold=975,
    max_pruned_weights=20
)

print("\n🎉 DNA Browser ready!")
print("Try the following:")
print("  1. Change the 'Condition' dropdown to view different conditions")
print("  2. Switch 'Show' dropdown to see network topology")
print("  3. Look for orange highlighting in voltage traces")
print("  4. Notice the 🎯 symbols on criteria neurons")

# Display the browser
display(browser)

## Summary - REAL Simulation Test

This test demonstrates the enhanced DNA browser functionality using the `new_jh_weights` DNA vector with **REAL simulation data** (no artificial noise). The key improvements include:

### ✨ Enhanced Features
- **REAL missed scoring visualization**: Orange highlights show exactly where points were lost in actual simulation
- **Authentic diagnostic data**: Uses real `diagnose_conditions()` output from simulation
- **Condition selection**: View experimental, control, or both conditions
- **Criteria neuron highlighting**: 🎯 symbols mark neurons used for fitness evaluation
- **Pure simulation dynamics**: No artificial noise - authentic Izhikevich neuron behavior

### 🔬 Technical Implementation
- **Real simulation engine**: Uses `run_dna_with_voltage_tracking()` from dna_simulation.py
- **Data format compliance**: Exactly matches analyze_multiple_ga_results.ipynb structure
- **Authentic spike rasters**: Real spike timing from simulation, not mock data
- **Interactive visualization**: Plotly-based plots with hover information

### 📊 new_jh_weights Performance
The `new_jh_weights` DNA vector represents a hand-crafted network with 18 connections:
- **Real experimental score**: {simulation_result['experimental']['score']} (missed {len(simulation_result['experimental']['missed_points'])} points)
- **Real control score**: {simulation_result['control']['score']} (missed {len(simulation_result['control']['missed_points'])} points)  
- **Total score**: {simulation_result['experimental']['score'] + simulation_result['control']['score']}
- **Network coverage**: Spans key basal ganglia pathways (Somat → ALM → BG → VM)

This test validates that the missed scoring visualization works correctly with real simulation data, providing an authentic view of where and why DNA vectors lose fitness points during evaluation.